In [1]:
import os
import sqlite3
from typing import Optional

import pandas as pd
import gradio as gr
from pydantic import BaseModel, ValidationError, StrictInt, StrictFloat, StrictStr, field_validator


# =========================
# 1) 기본 설정
# =========================
DB_NAME = "customer_users.db"
CSV_FILE = "customers(1).csv"


# =========================
# 2) Pydantic 모델
#    - 타입 오류가 나면 바로 ValidationError 발생
# =========================
class UserCreate(BaseModel):
    customer_id: StrictStr
    gender: StrictStr
    payment_method: StrictStr
    residence: StrictStr
    membership_grade: StrictStr
    satisfaction: StrictInt
    recent_access_hour: StrictInt
    preferred_temp: StrictFloat
    age: StrictInt
    quantity: StrictInt
    total_payment: StrictInt

    @field_validator("customer_id", "gender", "payment_method", "residence", "membership_grade")
    @classmethod
    def not_empty_string(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("빈 문자열은 허용되지 않습니다.")
        return v

    @field_validator("satisfaction")
    @classmethod
    def validate_satisfaction(cls, v: int) -> int:
        if not (1 <= v <= 5):
            raise ValueError("만족도는 1~5 범위여야 합니다.")
        return v

    @field_validator("recent_access_hour")
    @classmethod
    def validate_hour(cls, v: int) -> int:
        if not (0 <= v <= 23):
            raise ValueError("최근접속시간(시)은 0~23 범위여야 합니다.")
        return v

    @field_validator("age", "quantity", "total_payment")
    @classmethod
    def non_negative_int(cls, v: int) -> int:
        if v < 0:
            raise ValueError("음수는 허용되지 않습니다.")
        return v


class UserUpdate(BaseModel):
    gender: Optional[StrictStr] = None
    payment_method: Optional[StrictStr] = None
    residence: Optional[StrictStr] = None
    membership_grade: Optional[StrictStr] = None
    satisfaction: Optional[StrictInt] = None
    recent_access_hour: Optional[StrictInt] = None
    preferred_temp: Optional[StrictFloat] = None
    age: Optional[StrictInt] = None
    quantity: Optional[StrictInt] = None
    total_payment: Optional[StrictInt] = None

    @field_validator("gender", "payment_method", "residence", "membership_grade")
    @classmethod
    def not_empty_optional_string(cls, v):
        if v is not None and not v.strip():
            raise ValueError("빈 문자열은 허용되지 않습니다.")
        return v

    @field_validator("satisfaction")
    @classmethod
    def validate_satisfaction(cls, v):
        if v is not None and not (1 <= v <= 5):
            raise ValueError("만족도는 1~5 범위여야 합니다.")
        return v

    @field_validator("recent_access_hour")
    @classmethod
    def validate_hour(cls, v):
        if v is not None and not (0 <= v <= 23):
            raise ValueError("최근접속시간(시)은 0~23 범위여야 합니다.")
        return v

    @field_validator("age", "quantity", "total_payment")
    @classmethod
    def non_negative_int(cls, v):
        if v is not None and v < 0:
            raise ValueError("음수는 허용되지 않습니다.")
        return v


# =========================
# 3) DB 연결 / 초기화
# =========================
def get_conn():
    return sqlite3.connect(DB_NAME)


def create_table():
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS users (
            customer_id TEXT PRIMARY KEY,
            gender TEXT NOT NULL,
            payment_method TEXT NOT NULL,
            residence TEXT NOT NULL,
            membership_grade TEXT NOT NULL,
            satisfaction INTEGER NOT NULL,
            recent_access_hour INTEGER NOT NULL,
            preferred_temp REAL NOT NULL,
            age INTEGER NOT NULL,
            quantity INTEGER NOT NULL,
            total_payment INTEGER NOT NULL
        )
    """)
    conn.commit()
    conn.close()


def seed_from_csv():
    if not os.path.exists(CSV_FILE):
        return f"CSV 파일이 없어 초기 적재를 건너뜁니다: {CSV_FILE}"

    df = pd.read_csv(CSV_FILE)
    df = df.rename(columns={
        "고객ID": "customer_id",
        "성별": "gender",
        "결제수단": "payment_method",
        "거주지": "residence",
        "회원등급": "membership_grade",
        "만족도": "satisfaction",
        "최근접속시간(시)": "recent_access_hour",
        "선호제품군_적정온도": "preferred_temp",
        "나이": "age",
        "구매수량": "quantity",
        "총결제금액": "total_payment"
    })

    required_cols = [
        "customer_id", "gender", "payment_method", "residence", "membership_grade",
        "satisfaction", "recent_access_hour", "preferred_temp", "age", "quantity", "total_payment"
    ]
    df = df[required_cols]

    # Pydantic으로 CSV 데이터도 한 번 검증
    valid_rows = []
    for _, row in df.iterrows():
        try:
            validated = UserCreate.model_validate({
                "customer_id": row["customer_id"],
                "gender": row["gender"],
                "payment_method": row["payment_method"],
                "residence": row["residence"],
                "membership_grade": row["membership_grade"],
                "satisfaction": int(row["satisfaction"]),
                "recent_access_hour": int(row["recent_access_hour"]),
                "preferred_temp": float(row["preferred_temp"]),
                "age": int(row["age"]),
                "quantity": int(row["quantity"]),
                "total_payment": int(row["total_payment"]),
            })
            valid_rows.append(validated)
        except ValidationError:
            continue

    conn = get_conn()
    cur = conn.cursor()
    cur.executemany("""
        INSERT OR REPLACE INTO users (
            customer_id, gender, payment_method, residence, membership_grade,
            satisfaction, recent_access_hour, preferred_temp, age, quantity, total_payment
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, [
        (
            u.customer_id, u.gender, u.payment_method, u.residence, u.membership_grade,
            u.satisfaction, u.recent_access_hour, u.preferred_temp, u.age, u.quantity, u.total_payment
        )
        for u in valid_rows
    ])
    conn.commit()
    conn.close()
    return f"초기 적재 완료: {len(valid_rows)}건"


# =========================
# 4) CRUD 함수
# =========================
def create_user(
    customer_id, gender, payment_method, residence, membership_grade,
    satisfaction, recent_access_hour, preferred_temp, age, quantity, total_payment
):
    try:
        user = UserCreate.model_validate({
            "customer_id": customer_id,
            "gender": gender,
            "payment_method": payment_method,
            "residence": residence,
            "membership_grade": membership_grade,
            "satisfaction": satisfaction,
            "recent_access_hour": recent_access_hour,
            "preferred_temp": preferred_temp,
            "age": age,
            "quantity": quantity,
            "total_payment": total_payment
        })
    except ValidationError as e:
        # 타입 오류 / 범위 오류 시 바로 거절
        raise gr.Error(f"입력 타입 또는 값 오류:\n{e}")

    conn = get_conn()
    cur = conn.cursor()

    # 중복 확인
    cur.execute("SELECT customer_id FROM users WHERE customer_id = ?", (user.customer_id,))
    if cur.fetchone():
        conn.close()
        raise gr.Error("이미 존재하는 customer_id 입니다.")

    cur.execute("""
        INSERT INTO users (
            customer_id, gender, payment_method, residence, membership_grade,
            satisfaction, recent_access_hour, preferred_temp, age, quantity, total_payment
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        user.customer_id, user.gender, user.payment_method, user.residence, user.membership_grade,
        user.satisfaction, user.recent_access_hour, user.preferred_temp, user.age, user.quantity, user.total_payment
    ))
    conn.commit()
    conn.close()
    return f"생성 완료: {user.customer_id}"


def read_user(customer_id):
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("""
        SELECT customer_id, gender, payment_method, residence, membership_grade,
               satisfaction, recent_access_hour, preferred_temp, age, quantity, total_payment
        FROM users
        WHERE customer_id = ?
    """, (customer_id,))
    row = cur.fetchone()
    conn.close()

    if not row:
        return pd.DataFrame(columns=[
            "customer_id", "gender", "payment_method", "residence", "membership_grade",
            "satisfaction", "recent_access_hour", "preferred_temp", "age", "quantity", "total_payment"
        ])

    return pd.DataFrame([row], columns=[
        "customer_id", "gender", "payment_method", "residence", "membership_grade",
        "satisfaction", "recent_access_hour", "preferred_temp", "age", "quantity", "total_payment"
    ])


def read_all(limit):
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("""
        SELECT customer_id, gender, payment_method, residence, membership_grade,
               satisfaction, recent_access_hour, preferred_temp, age, quantity, total_payment
        FROM users
        ORDER BY customer_id
        LIMIT ?
    """, (limit,))
    rows = cur.fetchall()
    conn.close()

    return pd.DataFrame(rows, columns=[
        "customer_id", "gender", "payment_method", "residence", "membership_grade",
        "satisfaction", "recent_access_hour", "preferred_temp", "age", "quantity", "total_payment"
    ])


def update_user(
    customer_id, gender, payment_method, residence, membership_grade,
    satisfaction, recent_access_hour, preferred_temp, age, quantity, total_payment
):
    try:
        payload = UserUpdate.model_validate({
            "gender": gender,
            "payment_method": payment_method,
            "residence": residence,
            "membership_grade": membership_grade,
            "satisfaction": satisfaction,
            "recent_access_hour": recent_access_hour,
            "preferred_temp": preferred_temp,
            "age": age,
            "quantity": quantity,
            "total_payment": total_payment
        })
    except ValidationError as e:
        raise gr.Error(f"수정 입력 오류:\n{e}")

    update_data = payload.model_dump(exclude_none=True)
    if not update_data:
        raise gr.Error("수정할 값이 없습니다.")

    conn = get_conn()
    cur = conn.cursor()

    cur.execute("SELECT customer_id FROM users WHERE customer_id = ?", (customer_id,))
    if not cur.fetchone():
        conn.close()
        raise gr.Error("수정 대상 customer_id가 존재하지 않습니다.")

    set_clause = ", ".join([f"{col} = ?" for col in update_data.keys()])
    values = list(update_data.values()) + [customer_id]

    cur.execute(f"UPDATE users SET {set_clause} WHERE customer_id = ?", values)
    conn.commit()
    conn.close()

    return f"수정 완료: {customer_id}"


def delete_user(customer_id):
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("SELECT customer_id FROM users WHERE customer_id = ?", (customer_id,))
    if not cur.fetchone():
        conn.close()
        raise gr.Error("삭제 대상 customer_id가 존재하지 않습니다.")

    cur.execute("DELETE FROM users WHERE customer_id = ?", (customer_id,))
    conn.commit()
    conn.close()
    return f"삭제 완료: {customer_id}"


# =========================
# 5) 앱 시작 시 초기화
# =========================
create_table()
INIT_MSG = seed_from_csv()


# =========================
# 6) Gradio UI
# =========================
with gr.Blocks(title="Customer CRUD Service") as demo:
    gr.Markdown("# Customer CRUD Service")
    gr.Markdown(f"초기 상태: {INIT_MSG}")

    with gr.Tab("Create"):
        c_customer_id = gr.Textbox(label="customer_id")
        c_gender = gr.Textbox(label="gender")
        c_payment_method = gr.Textbox(label="payment_method")
        c_residence = gr.Textbox(label="residence")
        c_membership_grade = gr.Textbox(label="membership_grade")
        c_satisfaction = gr.Number(label="satisfaction (int)")
        c_recent_access_hour = gr.Number(label="recent_access_hour (0~23)")
        c_preferred_temp = gr.Number(label="preferred_temp (float)")
        c_age = gr.Number(label="age (int)")
        c_quantity = gr.Number(label="quantity (int)")
        c_total_payment = gr.Number(label="total_payment (int)")
        c_btn = gr.Button("생성")
        c_out = gr.Textbox(label="결과")

        c_btn.click(
            fn=create_user,
            inputs=[
                c_customer_id, c_gender, c_payment_method, c_residence, c_membership_grade,
                c_satisfaction, c_recent_access_hour, c_preferred_temp, c_age, c_quantity, c_total_payment
            ],
            outputs=c_out
        )

    with gr.Tab("Read"):
        with gr.Row():
            r_customer_id = gr.Textbox(label="customer_id")
            r_btn = gr.Button("단건 조회")
        r_out = gr.Dataframe(label="조회 결과")

        with gr.Row():
            r_limit = gr.Number(value=10, label="limit")
            r_all_btn = gr.Button("전체 조회")
        r_all_out = gr.Dataframe(label="전체 결과")

        r_btn.click(fn=read_user, inputs=r_customer_id, outputs=r_out)
        r_all_btn.click(fn=read_all, inputs=r_limit, outputs=r_all_out)

    with gr.Tab("Update"):
        u_customer_id = gr.Textbox(label="customer_id")
        u_gender = gr.Textbox(label="gender")
        u_payment_method = gr.Textbox(label="payment_method")
        u_residence = gr.Textbox(label="residence")
        u_membership_grade = gr.Textbox(label="membership_grade")
        u_satisfaction = gr.Number(label="satisfaction (int)")
        u_recent_access_hour = gr.Number(label="recent_access_hour (0~23)")
        u_preferred_temp = gr.Number(label="preferred_temp (float)")
        u_age = gr.Number(label="age (int)")
        u_quantity = gr.Number(label="quantity (int)")
        u_total_payment = gr.Number(label="total_payment (int)")
        u_btn = gr.Button("수정")
        u_out = gr.Textbox(label="결과")

        u_btn.click(
            fn=update_user,
            inputs=[
                u_customer_id, u_gender, u_payment_method, u_residence, u_membership_grade,
                u_satisfaction, u_recent_access_hour, u_preferred_temp, u_age, u_quantity, u_total_payment
            ],
            outputs=u_out
        )

    with gr.Tab("Delete"):
        d_customer_id = gr.Textbox(label="customer_id")
        d_btn = gr.Button("삭제")
        d_out = gr.Textbox(label="결과")

        d_btn.click(fn=delete_user, inputs=d_customer_id, outputs=d_out)


if __name__ == "__main__":
    # 임시 공개 링크
    demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://13c0da1be1574cdb86.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
